# 1. Dataset Audit

## 1.1 Dataset Acquisition and Loading

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import warnings

LABEL_COL = "Label"
warnings.filterwarnings('ignore')

PROJECT_DIR = "/Users/ashishkrsingh/Developers/Machine Learning/Projects/Network-Intrusion-Detection"
REPORT_DIR = Path(PROJECT_DIR) / "reports"

print("Import completed successfully")
print("Setup completed successfully")

In [ ]:
DATASETS = []

RAW_DATA_DIR = Path(PROJECT_DIR) / "data" / "raw"

print(f"{"="*15} Loading from {RAW_DATA_DIR} {"="*15}\n")
for path in RAW_DATA_DIR.glob("*.csv"):
    try:
        df = pd.read_csv(path)
        df.columns = df.columns.str.strip()
        df["source_file"] = path.name.split('.')[0]
        print(f"Loaded {path.parts[-1]} ({path.stat().st_size / (1024 ** 2):.2f} MB)")
        DATASETS.append(df)
    except Exception as e:
        print(f"Error while loading {path.parts[-1]}: {e}")

print(f"\n{"="*60} Loading Complete {"="*60}")
print(f"\nTotal datasets loaded: {len(DATASETS)}")

COMBINED_DATASET = pd.concat(DATASETS, ignore_index=True)
print(f"Combined dataset successfully created")
TOTAL_MEMORY = COMBINED_DATASET.memory_usage(deep=True).sum()
print(f"Total memory used: {TOTAL_MEMORY / (1024 ** 2):.2f} MB")

In [ ]:
COLS = COMBINED_DATASET.columns.tolist()
DATA_TYPES = COMBINED_DATASET.dtypes.to_list()
LABEL_COLS = COMBINED_DATASET[LABEL_COL].unique().tolist()

for DATASET in DATASETS:
    INDIVIDUAL_COLS = DATASET.columns.tolist()
    INDIVIDUAL_DATA_TYPES = DATASET.dtypes.to_list()
    INDIVIDUAL_LABEL_COLS = DATASET[LABEL_COL].unique().tolist()

    if len(set(COLS) - set(INDIVIDUAL_COLS)) > 0:
        print(f"\nColumn mismatch in {DATASET['source_file'].iloc[0]}")
        print(f"Columns in mismatched: {set(COLS) - set(INDIVIDUAL_COLS)}")

    if len(set(DATA_TYPES) - set(INDIVIDUAL_DATA_TYPES)) > 0:
        print(f"\nData type mismatch in {DATASET['source_file'].iloc[0]}")
        print(f"Data types mismatched: {set(DATA_TYPES) - set(INDIVIDUAL_DATA_TYPES)}")

    if len(set(LABEL_COLS) - set(INDIVIDUAL_LABEL_COLS)) > 0:
        print(f"\nLabel values mismatch in {DATASET['source_file'].iloc[0]}")
        print(f"Label values mismatched - {len(set(LABEL_COLS)-set(INDIVIDUAL_LABEL_COLS))}")
        for label in set(LABEL_COLS) - set(INDIVIDUAL_LABEL_COLS):
            print(f" - {label}")

    else:
        print(f"No column, data type mismatch in {DATASET['source_file'].iloc[0]}")

In [ ]:
for DATASET in DATASETS:
    for cols in COLS:
        if cols != LABEL_COL and cols != "source_file":
            CURR_COL_DATA_TYPE = COMBINED_DATASET[cols].dtype
            INDIVIDUAL_DATA_TYPES = DATASET[cols].dtype
            if len(set([CURR_COL_DATA_TYPE]) - set([INDIVIDUAL_DATA_TYPES])) > 0:
                print(f"\nData type mismatch in {DATASET['source_file'].iloc[0]} for column {cols}")
                print(f"Data type mismatched: {len(set([CURR_COL_DATA_TYPE]) - set([INDIVIDUAL_DATA_TYPES]))}")
                for data_type in set([CURR_COL_DATA_TYPE]) - set([INDIVIDUAL_DATA_TYPES]):
                    print(f" - {data_type}")
            else:
                print(f"No data type mismatch in {DATASET['source_file'].iloc[0]} for column {cols}")

## 1.2 Dataset Structure and Composition

In [ ]:
COMBINED_DATASET.sample(n = 10, random_state=42)

In [ ]:
print(f"Dataset shape: {COMBINED_DATASET.shape}")
print(f"# rows: {COMBINED_DATASET.shape[0]}, # columns: {COMBINED_DATASET.shape[1]}\n")

COLS = COMBINED_DATASET.columns.tolist()

if len(COLS) != len(set(COLS)):
    print(f"Duplicate columns found: {len(COLS) - len(set(COLS))}")
else:
    print("No duplicate columns found\n")

NUMERIC_COLS = COMBINED_DATASET.select_dtypes(include=[np.number]).columns.tolist()
print(f"# Numerical columns: {len(NUMERIC_COLS)}")

CATEGORICAL_COLS = COMBINED_DATASET.select_dtypes(include=['object']).columns.tolist()
print(f"# Categorical columns: {len(CATEGORICAL_COLS) - 1}")
for cols in CATEGORICAL_COLS:
    if cols != 'source_file':
        if cols == LABEL_COL and cols != 'source_file':
            print(f" - {cols} (target column)")
        else:
            print(f" - {cols}")

print(f"\nMemory usage: {COMBINED_DATASET.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")
        

In [ ]:
for col in NUMERIC_COLS:
    print(f"\n{'='*5} Column: {col} {'='*5}\n")
    print(f" - Min: {COMBINED_DATASET[col].min()}")
    print(f" - Max: {COMBINED_DATASET[col].max()}")
    print(f" - Mean: {COMBINED_DATASET[col].mean()}")
    print(f" - Median: {COMBINED_DATASET[col].median()}")
    print(f" - Std: {COMBINED_DATASET[col].std()}")

In [ ]:
CONSTANT_COLS = []
for cols in NUMERIC_COLS:
    if COMBINED_DATASET[cols].unique().shape[0] == 1:
        CONSTANT_COLS.append(cols)

if len(CONSTANT_COLS) > 0:
    print(f"Constant columns found: {len(CONSTANT_COLS)}")
    for cols in CONSTANT_COLS:
        print(f" - {cols}")
else:
    print("No constant columns found")

In [ ]:
INVALID_VALUE_COLS = []

for cols in NUMERIC_COLS:
    if (COMBINED_DATASET[cols] < 0).any():
        INVALID_VALUE_COLS.append(cols)

if len(INVALID_VALUE_COLS) > 0:
    print(f"Columns with invalid values found: {len(INVALID_VALUE_COLS)}")
    for cols in INVALID_VALUE_COLS:
        print(f" - {cols}")
else:
    print("No columns with invalid values found")

In [ ]:
LABEL_COLS = COMBINED_DATASET[LABEL_COL].unique().tolist()

print(f"# Unique Label values: {len(LABEL_COLS)}: {COMBINED_DATASET[LABEL_COL].shape[0]}")
for label in LABEL_COLS:
    print(f" - {label}: {COMBINED_DATASET[COMBINED_DATASET[LABEL_COL] == label].shape[0]},{COMBINED_DATASET[COMBINED_DATASET[LABEL_COL] == label].shape[0] / 
                                                                                           COMBINED_DATASET.shape[0] * 100:.4f}%")

## 1.3 Data Completeness and Integrity

### 1.3.1 Checking Missing Values

In [ ]:
for DATASET in DATASETS:
    NAN_COUNTS = DATASET.isnull().sum()
    NAN_PCT = DATASET.isnull().mean() * 100

    NAN_df = pd.DataFrame({'NaN Count': NAN_COUNTS, 'NaN %': NAN_PCT})
    NAN_df = NAN_df[NAN_df['NaN Count'] > 0].sort_values(by='NaN Count', ascending=False)

    if NAN_df.empty:
        print(f"\n{'='*5} No NaN values in {DATASET['source_file'].iloc[0]} {'='*5}")

    else:
        print(f"\n{'='*6} {DATASET['source_file'].iloc[0]} {'='*6}")
        display(NAN_df)

In [ ]:
NAN_COUNTS = COMBINED_DATASET.isnull().sum()
NAN_PCT = COMBINED_DATASET.isnull().mean() * 100

nan_df = pd.DataFrame({"NAN counts": NAN_COUNTS, "NAN %": NAN_PCT})
nan_df = nan_df[nan_df['NAN counts'] > 0].sort_values('NAN counts', ascending=False)

if nan_df.empty:
    print("\nNo missing values found in the dataset.")
else:
    print(f"# Columns with missing values: {nan_df.shape[0]}\n")
    display(nan_df)


### 1.3.2 Checking Infinite Values

In [ ]:
for DATASET in DATASETS:
    INF_COUNTS = DATASET.isin([np.inf, -np.inf]).sum()
    INF_PCT = DATASET.isin([np.inf, -np.inf]).mean() * 100

    INF_df = pd.DataFrame({'Inf Count': INF_COUNTS, 'Inf %': INF_PCT})
    INF_df = INF_df[INF_df['Inf Count'] > 0].sort_values(by='Inf Count', ascending=False)

    if INF_df.empty:
        print(f"\n{'='*5} No Inf values in {DATASET['source_file'].iloc[0]} {'='*5}")

    else:
        print(f"\n{'='*6} {DATASET['source_file'].iloc[0]} {'='*6}")
        display(INF_df)

In [ ]:
INF_COUNTS = COMBINED_DATASET.isin([np.inf, -np.inf]).sum()
INF_PCT = COMBINED_DATASET.isin([np.inf, -np.inf]).mean() * 100

inf_df = pd.DataFrame({"Inf counts": INF_COUNTS, "Inf %": INF_PCT})
inf_df = inf_df[inf_df['Inf counts'] > 0].sort_values('Inf counts', ascending=False)

if inf_df.empty:
    print("\nNo infinite values found in the dataset.")
else:
    print(f"# Columns with infinite values: {inf_df.shape[0]}\n")
    display(inf_df)

### 1.3.3 Checking Duplicate Values

In [ ]:
TOTAL_DUPS = 0

for DATASET in DATASETS:
    DUP_COUNTS = DATASET.duplicated().sum()
    DUP_PCT = DATASET.duplicated().mean() * 100

    DUP_BY_LABEL = (
    DATASET[DATASET.duplicated()]
    .groupby(LABEL_COL)
    .size()
    .sort_values(ascending=False)
    .rename('Dup Count')
    )

    if DUP_BY_LABEL.empty:
        print(f"\n{'='*5} No duplicate values in {DATASET['source_file'].iloc[0]} {'='*5}")

    else:
        print(f"\n{'='*6} {DATASET['source_file'].iloc[0]} {'='*6}\n")
        print(f'Total duplicate rows : {DUP_COUNTS:,}')
        TOTAL_DUPS += DUP_COUNTS
        print(f'Duplicate percentage  : {DUP_PCT:.2f}%\n')
        print(DUP_BY_LABEL)     

print("="*65)
print(f"\nTotal duplicate rows across all datasets: {TOTAL_DUPS}")

In [ ]:
DUP_COUNTS = COMBINED_DATASET.duplicated().sum()
DUP_PCT = COMBINED_DATASET.duplicated().mean() * 100

print(f'Total duplicate rows : {DUP_COUNTS:,}')
print(f'Duplicate percentage  : {DUP_PCT:.2f}%')

dup_by_label = (
    COMBINED_DATASET[COMBINED_DATASET.duplicated()]
    .groupby(LABEL_COL)
    .size()
    .sort_values(ascending=False)
    .rename('Dup Count')
)

print(f"\n{'='*5} Duplicate rows by Label {'='*5}")
print(dup_by_label)

In [ ]:
AUDIT_DATA = DATASET.drop(columns=['source_file'])

DUP_COUNTS = AUDIT_DATA.duplicated(keep='first').sum()
DUP_PCT = AUDIT_DATA.duplicated().mean() * 100

print(f'Total duplicate rows : {DUP_COUNTS:,}')
print(f'Duplicate percentage  : {DUP_PCT:.2f}%')

dup_by_label = (
    AUDIT_DATA[AUDIT_DATA.duplicated()]
    .groupby(LABEL_COL)
    .size()
    .sort_values(ascending=False)
    .rename('Dup Count')
)

print(f"\n{'='*5} Duplicate rows by Label {'='*5}")
print(dup_by_label)